# LLM Laika
Antes de rodar: **Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)**.

Tudo (livros, artigos, índice de busca, pré-treino, fine-tuning e memória) fica salvo no seu
Google Drive, na pasta `LLM_Laika`. Se a sessão cair, nada se perde.

| Quero... | Rode as células |
|---|---|
| Fazer tudo do zero | 1, 2, 3, 4, 5 |
| **Só refazer o fine-tuning** (usando o pré-treino que já está no Drive) | 1, 4, 5 |
| Atualizar livros/artigos e o índice de busca | 1, 2 |
| Só conversar | 1, 5 |
| Levar o modelo para o celular (app Laika) | 1, 6 |
| **Ligar as células pelo app, sem deixar o Colab aberto** | 0 (uma vez só) |

Os campos ao lado das células (MODO, EPOCAS...) são os mesmos que o app mostra na tela **Colab**.
Pelo app, a célula 1 sempre roda antes das outras; as células 0 e 5 só funcionam aqui no navegador.

In [ ]:
# 0) LIGAR O APP DA LAIKA AO COLAB [só no navegador] — rode uma vez só
# Depois disso, o app (menu ⋮ → Colab) liga as células 1, 2, 3, 4 e 6 sem você abrir o Colab:
# o GitHub Actions do repositório roda o Colab CLI com a sua conta e desliga a máquina no fim.
# Antes de rodar:
#  a) crie um token em https://github.com/settings/personal-access-tokens/new (tipo "fine-grained"),
#     só para o repositório LLM_Laika, com as permissões Actions (leitura e escrita),
#     Secrets (leitura e escrita) e Contents (leitura). Guarde o token: ele também vai no app;
#  b) o arquivo .github/workflows/colab.yml precisa estar no branch principal do repositório.
# Ao rodar: abra o link que aparecer, entre com a sua conta Google, copie o código e cole aqui.
REPOSITORIO = "BrunoDevPro/LLM_Laika"  #@param {type:"string"}
import os, json, base64, getpass, urllib.request, urllib.error

escopos = ",".join(["openid", "https://www.googleapis.com/auth/cloud-platform",
                    "https://www.googleapis.com/auth/userinfo.email",
                    "https://www.googleapis.com/auth/colaboratory"])
!gcloud auth application-default login --no-launch-browser --scopes={escopos}
adc = os.path.join(os.environ.get("CLOUDSDK_CONFIG") or os.path.expanduser("~/.config/gcloud"),
                   "application_default_credentials.json")
if not os.path.exists(adc):
    raise RuntimeError("O login do Google não terminou. Rode a célula de novo.")

!pip install -q pynacl
from nacl import encoding, public
token = getpass.getpass("Token do GitHub (não aparece na tela): ").strip()

def github(metodo, caminho, dados=None):
    req = urllib.request.Request(
        f"https://api.github.com/repos/{REPOSITORIO}{caminho}", method=metodo,
        data=None if dados is None else json.dumps(dados).encode(),
        headers={"Authorization": "Bearer " + token, "Accept": "application/vnd.github+json",
                 "X-GitHub-Api-Version": "2022-11-28"})
    try:
        with urllib.request.urlopen(req) as r:
            corpo = r.read()
            return json.loads(corpo) if corpo else {}
    except urllib.error.HTTPError as e:
        dicas = {401: "o token está errado ou venceu",
                 403: "o token não tem a permissão necessária (veja o item a)",
                 404: "repositório não encontrado ou o token não tem acesso a ele"}
        raise RuntimeError(f"GitHub {e.code} em {caminho}: {dicas.get(e.code, e.reason)}") from None

chave = github("GET", "/actions/secrets/public-key")
selado = public.SealedBox(public.PublicKey(chave["key"].encode(), encoding.Base64Encoder()))
cifrado = base64.b64encode(selado.encrypt(open(adc, "rb").read())).decode()
github("PUT", "/actions/secrets/COLAB_ADC", {"encrypted_value": cifrado, "key_id": chave["key_id"]})
print("Pronto: a permissão do Colab foi guardada no GitHub (segredo COLAB_ADC).")
try:
    github("GET", "/actions/workflows/colab.yml")
    print("O workflow do Colab está no repositório. Agora é só usar a tela Colab do app.")
except RuntimeError:
    print("Atenção: o .github/workflows/colab.yml ainda não está no branch principal (faça o merge).")

In [ ]:
# 1) PREPARAR: GPU, Google Drive, código e atalhos para as pastas do Drive
RAMO = "main"  #@param {type:"string"}
import os, shlex, codecs, subprocess, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (só processador)")

def rodar(comando):
    """Roda um comando mostrando a saída aos poucos; se ele falhar, a célula para com erro."""
    print("$", comando, flush=True)
    p = subprocess.Popen(comando, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env=dict(os.environ, PYTHONUNBUFFERED="1"))
    texto = codecs.getincrementaldecoder("utf-8")("replace")
    while pedaco := p.stdout.read1(65536):
        print(texto.decode(pedaco), end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"o comando falhou (código {p.returncode}): {comando}")

if not os.path.ismount("/content/drive"):  # pelo app, o Drive já chega montado
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/LLM_Laika"

if not os.path.exists("/content/LLM_Laika"):
    rodar("git clone -q https://github.com/BrunoDevPro/LLM_Laika.git /content/LLM_Laika")
os.chdir("/content/LLM_Laika")
rodar(f"git fetch -q origin {shlex.quote(RAMO)} && git checkout -q -f -B {shlex.quote(RAMO)} FETCH_HEAD")
rodar("pip install -q tokenizers datasets")

# As pastas grandes ficam no Drive; aqui são só atalhos
for local, remoto in [("data/books", "books"), ("data/wiki", "wiki"), ("data/qa_auto", "qa_auto"),
                      ("data/rag", "rag"), ("checkpoints", "checkpoints")]:
    remoto = f"{DRIVE}/{remoto}"
    os.makedirs(remoto, exist_ok=True)
    if not os.path.islink(local):
        if os.path.isdir(local) and os.listdir(local):
            rodar(f"cp -rn {local}/. {remoto}/")
        rodar(f"rm -rf {local}")
        os.symlink(remoto, local)

print("No Drive:", {p: len(os.listdir(f"{DRIVE}/{p}")) for p in ["books", "wiki", "rag", "checkpoints"]})

In [ ]:
# 2) DADOS: livros do Gutenberg + artigos da Wikipédia + índice de busca (RAG)
# Só baixa o que ainda não está no Drive. A primeira vez demora (dezenas de minutos).
LIVROS = 500  #@param {type:"integer"}
ARTIGOS = 100000  #@param {type:"integer"}
REFAZER_INDICE = False  #@param {type:"boolean"}
rodar(f"python prepare_data.py --max-books {LIVROS} --wiki {ARTIGOS}")
rodar("python rag.py build" + (" --force" if REFAZER_INDICE else ""))

In [ ]:
# 3) PRÉ-TREINO (ensina português). Escolha o MODO:
#   "pular"     -> usa o pré-treino que já está no Drive (padrão; se não existir, treina do zero)
#   "continuar" -> continua o pré-treino do Drive com os dados atuais (mesmo tokenizer)
#   "do_zero"   -> apaga e treina tudo de novo (necessário para o tokenizer aprender palavras novas)
# PASSOS = 0 usa o número de passos do preset. Trocar o PRESET exige "do_zero".
MODO = "pular"  #@param ["pular", "continuar", "do_zero"]
PRESET = "colab"  #@param ["tiny", "colab", "colab_big"]
PASSOS = 0  #@param {type:"integer"}
import os
extra = f" --max-steps {PASSOS}" if PASSOS > 0 else ""
existe = os.path.exists("checkpoints/pretrain_best.pt")
if MODO == "pular" and existe:
    print("Usando o pré-treino que já está no Drive.")
elif MODO == "continuar" and existe:
    rodar(f"python train.py --preset {PRESET} --resume{extra}")
else:
    rodar(f"python train.py --preset {PRESET}{extra}")

In [ ]:
# 4) FINE-TUNING (ensina a responder). Usa o pré-treino do Drive e grava checkpoints/finetuned.pt.
# Se o índice do RAG existir (célula 2), parte das perguntas é treinada com Contexto.
EPOCAS = 2  #@param {type:"integer"}
USAR_RAG = True  #@param {type:"boolean"}
rodar(f"python finetune.py --epochs {EPOCAS}" + ("" if USAR_RAG else " --no-rag"))

In [ ]:
# 5) CONVERSAR [só no navegador] (digite 'sair' para encerrar)
# Comandos: /fontes (mostra de onde veio o contexto), /rag (liga/desliga a busca), /memoria
# A Laika lembra o que você contar (ex.: "meu nome é Ana", "lembre que amanhã tenho prova").
# Para testar só o pré-treino: sys.argv = ["chat.py", "--base"]
import sys
sys.argv = ["chat.py"]
%run chat.py

In [ ]:
# 6) EXPORTAR PARA O CELULAR: um zip só, em Meu Drive/LLM_Laika/laika_pacote.zip
# No app: botão de versões → Escolher arquivos → laika_pacote.zip. Vira uma versão nova
# (a anterior fica nos backups). Pela tela Colab do app, o zip aparece sozinho em "Modelos prontos".
# INDICE (~640 MB) e TEXTOS (livros e artigos, para o celular montar o índice) deixam o zip bem maior.
MODELO = True  #@param {type:"boolean"}
INDICE = False  #@param {type:"boolean"}
TEXTOS = False  #@param {type:"boolean"}
import os, glob, zipfile, shutil
arquivos = []
if MODELO:
    arquivos += [("checkpoints/" + n, n) for n in ["finetuned.pt", "pretrain_best.pt", "tokenizer.json"]
                 if os.path.exists("checkpoints/" + n)]
if INDICE and os.path.exists("data/rag/laika_rag.db"):
    arquivos.append(("data/rag/laika_rag.db", "laika_rag.db"))
if TEXTOS:
    for p in ["books", "wiki", "modern"]:
        arquivos += [(f, p + "/" + os.path.basename(f)) for f in sorted(glob.glob(f"data/{p}/*.txt"))]
    if os.path.exists("data/books/titulos.tsv"):
        arquivos.append(("data/books/titulos.tsv", "books/titulos.tsv"))
if not arquivos:
    raise RuntimeError("Nada para exportar: rode o fine-tuning (célula 4) ou marque INDICE ou TEXTOS.")
with zipfile.ZipFile("/content/laika_pacote.zip", "w", zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for f, nome in arquivos:
        z.write(f, nome)
destino = DRIVE + "/laika_pacote.zip"
shutil.copy("/content/laika_pacote.zip", destino)
print(f"Pronto: {len(arquivos)} arquivos, {os.path.getsize(destino) / 2**20:.0f} MB em {destino}")